In [1]:
import config as cfg
from data_acquirer import get_data, gen_data
from data_processor import process
import heuristic
from solver import solve_pdpcr
from heuristic import Path

import copy
import datetime
from docplex.mp.model import Model
import heapq
import numpy as np
import time


In [169]:

class VietSolver:
    """Solve PDPCR applying Viet's idea
    """
    def __init__(self, processed, c_travel, c_operate, *kwargs):
        self.processed = processed
        self.depot = processed.depot
        self.tasks = processed.tasks
        self.c_travel = c_travel
        self.c_operate = c_operate
        self.tasks_remains = [task for task in processed.tasks]
        self.idx_task_mapper = {task.index: task for task in processed.tasks}
        
    def _all_tasks_finished(self):
        all_finished = False
        if len(self.tasks_remains) == 0:
            all_finished = True
        return all_finished
    
    def _cal_Jk(self,k):
        ns = [task.n for task in self.tasks_remains]
        k_largest = heapq.nlargest(k,ns)
        J_k = sum(k_largest)
        return J_k

    def _solve_viet(self, 
                    total_robots, 
                    tasks_each_iteration, 
                    consider_process_time):
        
        starting_times = {i:0 for i in range(total_robots)}
        J_k = self._cal_Jk(tasks_each_iteration)
        print(J_k)
        if J_k <= total_robots:
            self._lastest_solve_result = self._solve_third_case(total_robots, 
                                                                starting_times, 
                                                                tasks_each_iteration, 
                                                                consider_process_time)
        #TODO
    
    def _update_paths(self):
        """Update the paths of each robot as well as procesed time
        """
        for key, value in self._lastest_solve_result['sol'].get_value_dict(self._lastest_solve_result['x']).items():
            if value == 1:
                robot_idx = key[0]
                task_idx = key[1]
                self.paths[robot_idx].insert(self.idx_task_mapper[task_idx])

    
    def _remove_processed_tasks(self):
        for key, value in self._lastest_solve_result['sol'].get_value_dict(self._lastest_solve_result['x']).items():
            if value == 1:
                task_idx = key[1]
                procesed_task = self.idx_task_mapper[task_idx]
                #remove processed_task from self.tasks_remains
                try:
                    self.tasks_remains.remove(procesed_task)
                except:
                    pass
                # self.tasks_remains.pop(procesed_task)
    
    def _robot_returns(self):
        pass
        #TODO

    def solve(self, total_robots, tasks_each_iteration, consider_process_time):
        self.paths ={i:Path(self.depot) for i in range(total_robots)}
        while self._all_tasks_finished() == False:
            self._solve_viet(total_robots, tasks_each_iteration, consider_process_time)
            self._update_paths()
            self._remove_processed_tasks()
        self._robot_returns()
        #TODO
    
    def _construct_a(self, total_robots):
        """Construct a dictionary of cost from robot i to task j,

        Parameters
        ----------
        total_robots : int
            Total number of robots

        Returns
        -------
        a : dict
            a[i][j] is the cost to go from the current position of robot i
            (the destination of the last task that robot i processed) to
            pick up location of task j
        """
        if not hasattr(self,'paths'):
            self.paths ={i:Path(self.depot) for i in range(total_robots)}
        a = {}
        for i in range(total_robots):
            a[i] = self.c_travel[self.paths[i][-1].index]
        return a

    def _construct_a_artificial(self, total_robots,total_robots_required):
        a = {}
        a = self._construct_a(total_robots)
        for i in range(total_robots, total_robots_required):
            a[i] = {j:cfg.BIG_M1 for j in range(1,len(self.tasks_remains) + 1)}
        return a

    def _solve_second_case(self,
                           total_robots,
                           starting_times :dict,
                           tasks_each_iteration : int,
                           consider_process_time : bool):
        
        total_robots_required = np.sum([task.n for task in self.tasks_remains])
        A_idcs = [i for i in range(1, len(self.tasks_remains) + 1)]
        B_idcs = [j for j in range(total_robots)]
        Bp_idcs = [j for j in range(total_robots, len(total_robots_required))]
        BpA_idcs = [(i,j) for i in Bp_idcs for j in A_idcs]

        
        a = self._construct_a_artificial(total_robots, total_robots_required)
        t = starting_times
        c = self.c_operate
        n ={task.index: task.n for task in self.tasks_remains}

        m = Model('Viet_case_2')
        C_max = m.continuous_var(name = 'C_max')
        x = m.binary_var_dict(BpA_idcs,ub=1, name = 'x')
        v= m.binary_var_dict(A_idcs, name = 'v')
        q = m.binary_var_dict(Bp_idcs, name = 'q')
        d = m.continuous_var(name = 'd')

        m.minimize(C_max)
        if consider_process_time:
            m.add_constraints(t[i] + a[i][j]*(1 - v[j]) + c[j] <= C_max for i in B_idcs for j in A_idcs)
        else:
            m.add_constraints(t[i] + a[i][j]*(1 - v[j]) <= C_max for i in B_idcs for j in A_idcs)

        m.add_constraints(m.sum(x[i,j] for i in B_idcs) == n[j] for j in A_idcs)
        m.add_constraints(m.sum(x[i,j] for j in A_idcs) <= 1 for i in B_idcs)
        
        #TODO


    def _solve_third_case(self, 
                          total_robots, 
                          starting_times :dict, 
                          tasks_each_iteration : int, 
                          consider_process_time : bool):
        A_idcs = [i for i in range(1,len(self.tasks_remains) + 1)]
        B_idcs = [j for j in range(total_robots)]
        BA_idcs = [(i,j) for i in B_idcs for j in A_idcs]

        a = self._construct_a(total_robots)
        t = starting_times
        c = self.c_operate
        n ={task.index: task.n for task in self.tasks_remains}

        m = Model('Viet_case_1')

        C_max = m.continuous_var(name = 'C_max')
        x = m.binary_var_dict(BA_idcs,ub=1, name = 'x')

        m.minimize(C_max)

        if consider_process_time:
            m.add_constraints(t[i] + a[i][j]*x[i,j] +c[j] <= C_max for i in B_idcs for j in A_idcs)
        else:
            m.add_constraints(t[i] + a[i][j]*x[i,j] <= C_max for i in B_idcs for j in A_idcs)

        m.add_constraints(m.sum(x[i,j] for i in B_idcs) == n[j] for j in A_idcs)
        m.add_constraints(m.sum(x[i,j] for j in A_idcs) <= 1 for i in B_idcs)

        m.parameters.mip.tolerances.mipgap = cfg.SOLVE_GAP
        m.time_limit = cfg.SOLVE_TIME
        if cfg.EXPORT_MODEL == True:
            print(m.export_as_lp())
        sol = m.solve(log_output = cfg.LOG_OUTPUT)

        solved_gap = m.solve_details.mip_relative_gap
        processed_nodes = m.solve_details.nb_nodes_processed
        solve_time = m.solve_details.time
        solve_info = {
            'sol':sol,
            'x':x,
            'processed_nodes': processed_nodes,
            'gap':solved_gap,
            'solve_time':solve_time
        }
        
        return solve_info

In [165]:
#Test VietSolver
import helpers 

# first_names = ['LC1']
# sec_names = [4,6,8,10]
# third_names = [i for i in range(1,11)]
# data_paths =[f'lim/{j}00/{i}_{j}_{k}-5-uniform.csv'for i in first_names for j in sec_names for k in third_names]#
num_robots = [10*i for i in range(1,4)]
method = 5
file_type = 'csv'
is_depot_task = False

# data_path = 'lim/100/lc101-5-uniform.csv'
# data_path = 'data2.csv'
data_path = 'generated20.csv' #error

raw = get_data(data_path, file_type, is_depot_task)
processed_h = heuristic.process_greedy_1(raw)
depot = processed_h.depot
tasks = processed_h.tasks
c_travel = helpers.create_task_dist_dict(depot,tasks)
c_operate = helpers.create_operate_dict(depot,tasks)
tik = time.time()


In [174]:
num_robot = 40
task_each_iter = 1
# starting_times = {i:0 for i in range(num_robot)}
solve_viet = VietSolver(processed_h,c_travel, c_operate)
# solve_result = solve_viet._solve_viet(num_robot,task_each_iter,False)
# solve_viet._update_paths()
# solve_viet._remove_processed_tasks()
solve_result = solve_viet.solve(num_robot,task_each_iter,False)

2
C:\Users\CTNGUY~1\AppData\Local\Temp\Viet_case_1.lp
Version identifier: 22.1.0.0 | 2022-03-09 | 1a383f8ce
CPXPARAM_Read_DataCheck                          1
CPXPARAM_TimeLimit                               3600
CPXPARAM_MIP_Tolerances_MIPGap                   0
Tried aggregator 1 time.
Reduced MIP has 655 rows, 601 columns, and 2400 nonzeros.
Reduced MIP has 600 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (1.47 ticks)
Found incumbent of value 333.953007 after 0.02 sec. (4.75 ticks)
Probing time = 0.00 sec. (3.78 ticks)
Cover probing fixed 1 vars, tightened 3 bounds.
Tried aggregator 1 time.
Detecting symmetries...
MIP Presolve eliminated 600 rows and 1 columns.
Reduced MIP has 55 rows, 600 columns, and 1200 nonzeros.
Reduced MIP has 600 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (1.58 ticks)

Root node processing (before b&c):
  Real time             =    0.02 sec. (10.57 ticks)
Parallel b&c, 6 threads:
  Real time         

In [42]:
solve_result['sol'].get_value_dict(solve_result['x'])
# We have a dict in a form of {(0, 1): 1.0,
#  (0, 2): 0,
#  (0, 3): 0,
#  (1, 1): 1.0,
#  (1, 2): 0,
#  (1, 3): 0,
#  (2, 1): 0,
#  (2, 2): 1.0,
#  (2, 3): 0,
#  (3, 1): 0,
#  (3, 2): 0,
#  (3, 3): 1.0}
# The key is (robot_idx, task_idx) and the value is 1 if the robot i is assigned to task j
# Append to `paths` the index of the task that the robot i is assigned to
#initially paths ={i:[0] for i in range(total_robots)}
for key, value in solve_result['sol'].get_value_dict(solve_result['x']).items():
    if value == 1:
        robot_idx = key[0]
        task_idx = key[1]
        solve_viet.paths[robot_idx].append(task_idx)

{(0, 1): 1.0,
 (0, 2): 0,
 (0, 3): 0,
 (1, 1): 1.0,
 (1, 2): 0,
 (1, 3): 0,
 (2, 1): 0,
 (2, 2): 1.0,
 (2, 3): 0,
 (3, 1): 0,
 (3, 2): 0,
 (3, 3): 1.0}

In [23]:
sol['sol'].display()

solution for: Viet_case_1
objective: 200.853
status: OPTIMAL_SOLUTION(2)
C_max = 200.853
x_0_1 = 0.394
x_0_2 = 0.276
x_0_3 = 0.330
x_1_1 = 0.670
x_1_3 = 0.330
x_2_1 = 0.670
x_2_2 = 0.330
x_3_1 = 0.267
x_3_2 = 0.394
x_3_3 = 0.339


In [3]:
method = 5
data_name = 'generated3.csv' #real data

if False:
    is_depot_task = True
else:
    is_depot_task = False #TODO remove this

    raw = get_data(data_name,'csv', is_depot_task)

In [4]:

#WR2 data
sec_names = ['100','200','400','600','800','1000']
third_names = ['1','2','3','4','5']


data_paths =[f'wr2/bar-n{j}-{k}-3.csv'for j in sec_names for k in third_names]#TODO add data
dist_paths = [f'wr2/bar-n{j}-{k}-dist.csv'for j in sec_names for k in third_names]#TODO add data
num_robots = [5*i for i in range(2,3)]

method = 5
file_type = 'csv'
is_depot_task = False


In [2]:

#Li data

first_names = ['LC1']
sec_names = [4,6,8,10]
third_names = [i for i in range(1,11)]
data_paths =[f'lim/{j}00/{i}_{j}_{k}-5-uniform.csv'for i in first_names for j in sec_names for k in third_names]#
num_robots = [10*i for i in range(1,4)]
method = 5
file_type = 'csv'
is_depot_task = False

In [4]:
# CPLEX data li lim
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_data = process(raw, method)
            solve_result = solve_pdpcr(processed_data, num_robot, method)
            
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(solve_result['solve_time'])
            solve_log.append(solve_result['sol'].objective_value)
            solve_log.append(solve_result['processed_nodes'])
            solve_log.append(solve_result['gap'])
            now = datetime.datetime.now()
            solve_log.append(now.strftime("%Y-%m-%d %H:%M:%S"))

            with open('solve_log.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

Solving for lim/lc101-5-uniform.csv with 10 robots


KeyboardInterrupt: 

In [4]:
## Mass solve using heuristics
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            tik1 = time.time()
            solve_result_h = heuristic.solve_greedy_1(processed_h, num_robot)
            tok1 = time.time()
            greedy1_time = tok1 - tik1
            print(f'Greedy 1 time: {greedy1_time}')
            tik2 = time.time()
            solve_result_h_jp = heuristic.solve_greedy_jp(processed_h, num_robot)
            tok2 = time.time()
            greedy_jp_time = tok2 -tik2
            print(f'Greedy jp time: {greedy_jp_time}')
            tik3 = time.time()
            solve_result_h_jp_2 = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
            tok3 = time.time()
            greedy_jp_time_2 = tok3 -tik3
            print(f'Greedy jp time 2: {greedy_jp_time_2}')

        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(solve_result_h.get_makespan().round(2))
            solve_log.append(np.round(greedy1_time,2))
            solve_log.append(solve_result_h_jp.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time,2))
            solve_log.append(solve_result_h_jp_2.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time_2,2))
            with open('solve_log_h_2.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

Solving for lim/200/LC1_2_1-5-uniform.csv with 10 robots
Thao heuristics sol 858.06
Greedy 1 time: 0.08085489273071289
JP heuristics sol is 965.9
Greedy jp time: 4.004814386367798
JP heuristics sol is 935.59
Greedy jp time 2: 3.7341208457946777
Solving for lim/200/LC1_2_1-5-uniform.csv with 20 robots
Thao heuristics sol 515.02
Greedy 1 time: 0.08251357078552246
JP heuristics sol is 526.62
Greedy jp time: 4.056585788726807
JP heuristics sol is 524.33
Greedy jp time 2: 4.006733417510986
Solving for lim/200/LC1_2_1-5-uniform.csv with 30 robots
Thao heuristics sol 425.6
Greedy 1 time: 0.08888077735900879
JP heuristics sol is 408.42
Greedy jp time: 4.134079456329346
JP heuristics sol is 405.79
Greedy jp time 2: 3.984548330307007
Solving for lim/200/LC1_2_2-5-uniform.csv with 10 robots
Thao heuristics sol 1194.39
Greedy 1 time: 0.07666921615600586
JP heuristics sol is 1214.9
Greedy jp time: 3.8654634952545166
JP heuristics sol is 1324.86
Greedy jp time 2: 3.8144493103027344
Solving for lim/2

KeyboardInterrupt: 

In [3]:
import helpers
## Mass solve 2 opt
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_h = heuristic.process_greedy_1(raw)

            tik2 = time.time()
            solve_result_h2 = heuristic.solve_greedy_jp(processed_h, num_robot)
            tok2 = time.time()
            greedy_h2 = tok2 -tik2

            tik3 = time.time()
            solve_result_h3 = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
            tok3 = time.time()
            greedy_jp_time_2 = tok3 -tik3

            depot = processed_h.depot
            tasks = processed_h.tasks
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)
            tik = time.time()
            solve_opt = heuristic.TwoOptSolver(processed_h,c_travel, c_operate)
            graph,makespan_2opt = solve_opt.solve(total_robots= num_robot)
            tok = time.time()
            two_opt_time = tok - tik

            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(solve_result_h2.get_makespan().round(2))
            solve_log.append(np.round(greedy_h2,2))
            solve_log.append(solve_result_h3.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time_2,2))
            solve_log.append(makespan_2opt.round(2))
            solve_log.append(np.round(two_opt_time,2))
            now = datetime.datetime.now()
            solve_log.append(now.strftime("%Y-%m-%d %H:%M:%S"))
            with open('solve_log_h_2opt.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

Solving for lim/400/LC1_4_1-5-uniform.csv with 10 robots
JP heuristics sol is 1913.32
JP heuristics sol is 1798.18
JP heuristics sol is 1798.18
Initial makespan: 1798.1793001432543
Applied 2-opt successfully on edges (7, 105) and (10, 18), new makespan: 1791.1429748654325
Applied 2-opt successfully on edges (105, 141) and (50, 149), new makespan: 1789.5491559776592
Applied 2-opt successfully on edges (188, 10) and (7, 18), new makespan: 1787.921738395897
Solving for lim/400/LC1_4_1-5-uniform.csv with 20 robots
JP heuristics sol is 1010.4
JP heuristics sol is 1079.46
JP heuristics sol is 1079.46
Initial makespan: 1079.4615154578496
Applied 2-opt successfully on edges (92, 195) and (118, 148), new makespan: 1075.6728902373202
Solving for lim/400/LC1_4_1-5-uniform.csv with 30 robots
JP heuristics sol is 738.85
JP heuristics sol is 830.71
JP heuristics sol is 830.71
Initial makespan: 830.7123186806714
Applied 2-opt successfully on edges (136, 191) and (89, 119), new makespan: 830.654537340

In [40]:
data_path = 'generated17.csv'
num_robot = 2
raw = get_data(data_path, file_type, is_depot_task)
processed_data = process(raw, method)
processed_h = heuristic.process_greedy_1(raw)
tik1 = time.time()
solve_result_h_jp = heuristic.solve_greedy_jp(processed_h, num_robot)
solve_result_h_jp2 = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)

JP heuristics sol is 3636.02
JP heuristics sol is 3003.14


In [9]:
#Solve wr2
for data_path, dist_path in zip(data_paths, dist_paths):
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_data = process(raw, method)
            solve_result = solve_pdpcr(processed_data, num_robot, method, dist_path)
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(solve_result['solve_time'])
            solve_log.append(solve_result['sol'].objective_value)
            solve_log.append(solve_result['processed_nodes'])
            solve_log.append(solve_result['gap'])
            # add current time to solve_log:
            now = datetime.datetime.now()
            solve_log.append(now.strftime("%Y-%m-%d %H:%M:%S"))

            with open('solve_log.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)
        #write solve_log to a file:


solving with c_travel calculated from dist_matrix
C:\Users\CTNGUY~1\AppData\Local\Temp\VRP_ver_2_makespan.lp
Start solving vrp-based model for makespan obj.function using CPLEX
Version identifier: 22.1.0.0 | 2022-03-09 | 1a383f8ce
CPXPARAM_Read_DataCheck                          1
CPXPARAM_TimeLimit                               3600
CPXPARAM_MIP_Tolerances_MIPGap                   0
Tried aggregator 2 times.
MIP Presolve eliminated 82 rows and 125 columns.
MIP Presolve modified 275 coefficients.
Aggregator did 6 substitutions.
Reduced MIP has 3230 rows, 3119 columns, and 15385 nonzeros.
Reduced MIP has 3068 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (12.80 ticks)
Found incumbent of value 47.000000 after 0.02 sec. (19.55 ticks)
Probing time = 0.00 sec. (3.65 ticks)
Cover probing fixed 0 vars, tightened 152 bounds.
Tried aggregator 1 time.
Detecting symmetries...
MIP Presolve eliminated 55 rows and 0 columns.
MIP Presolve modified 6026 coefficients.
Reduce

In [ ]:
raw = get_data(data_path, file_type, is_depot_task)
processed_h = heuristic.process_greedy_1(raw)
depot = processed_h.depot
tasks = processed_h.tasks
c_travel = helpers.create_task_dist_dict(depot,tasks)
c_operate = helpers.create_operate_dict(depot,tasks)
tik = time.time()
solve_opt = heuristic.TwoOptSolver(processed_h,c_travel, c_operate)

In [16]:
#Get data

#Read file solve_log.csv, save it as a dataframe
import pandas as pd
solve_cplex_df = pd.read_csv('solve_log.csv', header=None)
solve_cplex_df.columns = ['data_path', 'num_robot', 'solve_time', 'objective_value', 'processed_nodes', 'gap', 'solve_date']

solve_h_df = pd.read_csv('solve_log_h.csv', header=None)
solve_h_df.columns = ['data_path', 'objective_value_1', 'time_greedy_1', 'objective_value_jp', 'time_greedy_jp']

In [18]:
# right join 2 dataframes on column data_path
merged_df = pd.merge(solve_cplex_df, solve_h_df, on='data_path', how='right')

In [48]:
merged_df['diff_h_1'] = np.round(1- merged_df['objective_value_1']/merged_df['objective_value'],2)
merged_df['diff_h_jp'] = np.round(1- merged_df['objective_value_jp']/merged_df['objective_value'],2)
#change order of columns in meged_df as follow: data_path, num_robot, objective_value, solve_time, objective_value_1, time_greedy_1, diff_h_1, objective_value_jp, time_greedy_jp, diff_h_jp,
merged_df = merged_df[['data_path', 'num_robot', 'objective_value', 'solve_time', 'objective_value_1', 'time_greedy_1', 'diff_h_1', 'objective_value_jp', 'time_greedy_jp', 'diff_h_jp']]
merged_df
#Remove the 'lim' prefix from data_path
merged_df['data_path'] = merged_df['data_path'].str.replace('lim/','')
#round solve_time to 2 decimal places
merged_df['solve_time'] = np.round(merged_df['solve_time'],2)
#round objective_value to 2 decimal places
merged_df['objective_value'] = np.round(merged_df['objective_value'],2)

In [50]:
merged_df.to_latex('merged_df.tex', index=True)

C:\Users\ctnguyen2\AppData\Local\Temp\ipykernel_7240\2685511736.py:1: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  merged_df.to_latex('merged_df.tex', index=True)


In [17]:
#Temp
import pandas as pd
solve_h_df = pd.read_csv('solve_log_h_2.csv', header=None)
solve_h_df.columns = ['data_path', 'objective_value_1', 'time_greedy_1', 'objective_value_2', 'time_greedy_2', 'objective_value_3', 'time_greedy_3']
#Keep the following columns: data_path, objective_value_2, time_greedy_2
solve_h_df = solve_h_df[['data_path', 'objective_value_2', 'time_greedy_2']]

In [3]:
solve_h_df_2opt = pd.read_csv('solve_log_h_2opt.csv', header=None)
solve_h_df_2opt.columns = ['data_path', 'num_robot', 'objective_value_3', 'time_greedy_3', 'objective_value_2opt', 'time_2opt']

In [21]:
#Join 2 dataframes on data_path
merged_df_2 = pd.merge(solve_h_df, solve_h_df_2opt, on='data_path', how='right')
#Change the order of columns as follow: data_path, num_robot, objective_value_2, time_greedy_2, objective_value_3, time_greedy_3, objective_value_2opt, time_2opt
merged_df_2 = merged_df_2[['data_path', 'num_robot', 'objective_value_2', 'time_greedy_2', 'objective_value_3', 'time_greedy_3', 'objective_value_2opt', 'time_2opt']]

In [22]:
#save the data as a latex table
merged_df_2.to_latex('merged_df_2.tex', index=True)

C:\Users\ctnguyen2\AppData\Local\Temp\ipykernel_12368\154469684.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  merged_df_2.to_latex('merged_df_2.tex', index=True)
